<a href="https://colab.research.google.com/github/lincoln-bot/Assignment-2-HASTS206-GROUP-6/blob/main/assignment2_demographic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#MEMBER 1
from google.colab import files
uploaded = files.upload()

Saving Assignment_2_Dataset.zip to Assignment_2_Dataset.zip


In [4]:
import zipfile
import os
zip_path = 'Assignment_2_Dataset.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
   zip_ref.extractall('assignment_data')
print('Data_set extracted successfully!')
os.listdir('assignment_data')


Data_set extracted successfully!


['age_sex.csv',
 'projection_scenarios.csv',
 'districts.csv',
 'insured_transitions.csv',
 'README.txt',
 'life_table_inputs.csv']

In [5]:
with open('assignment_data/README.txt', 'r') as file:
    readme = file.read()
print(readme)

HASTS206 | Assignment 2 | Dataset guide

All records are synthetic and describe hypothetical districts and insured lives
in a Zimbabwean teaching case. Dates and district labels are illustrative.
Extract this ZIP and keep the original CSV files for reproducibility.
Missing values are blank. Counts represent people unless stated otherwise.

districts.csv (24 rows, key district_id)
  population_2020 and population_2026 are opening stocks on 1 January.
  The opening 2025 stock is obtained by summing age_sex.csv.
  The four event counts cover calendar 2025 and close the balance at 1 Jan 2026.
  piped_water_pct is a district household percentage. Two values are missing.
  region is a synthetic reporting label.

age_sex.csv (864 rows, key district_id + age_start + sex)
  age_start gives 5-year bands (0 = 0-4, 5 = 5-9, ..., 85 = 85+).
  sex is F or M. population_2025 is the 1 Jan 2025 count.
  baseline_deaths_2025 is for residents present at baseline, classified by age
  at baseline. District

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import expm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
print('All Libraries imported successfully!')
districts = pd.read_csv('assignment_data/districts.csv')
age_sex = pd.read_csv('assignment_data/age_sex.csv')
life_table = pd.read_csv('assignment_data/life_table_inputs.csv')
scenarios = pd.read_csv('assignment_data/projection_scenarios.csv')
insured = pd.read_csv('assignment_data/insured_transitions.csv')
print('All five datasets loaded successfully!')

All Libraries imported successfully!
All five datasets loaded successfully!


In [7]:
import os, glob
import pandas as pd

folder = "assignment_data"
files = glob.glob(folder + "/*.csv")
data = {}
log = []

# 1. Load original datasets and inspect them
for file in files:
    name = os.path.basename(file)
    df = pd.read_csv(file)
    data[name] = df

    log.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Numeric Columns": sum(
            pd.api.types.is_numeric_dtype(df[c]) for c in df.columns
        )
    })

    print(f"\n{name}")
    print("Columns:", list(df.columns))
    print("Data types:\n", df.dtypes)

# 2. Save data-quality log
quality_log = pd.DataFrame(log)
os.makedirs("outputs", exist_ok=True)
quality_log.to_csv("outputs/before_analysis_quality_log.csv", index=False)

# 3. Check possible identifier columns
for name, df in data.items():
    for col in df.columns:
        if col.lower() in ["district_id", "district", "id"]:
            print(name, col, "Unique:", df[col].nunique(),
                  "Duplicates:", df[col].duplicated().sum())

# 4. Check numerical ranges
for name, df in data.items():
    print(f"\nNumerical ranges: {name}")
    print(df.select_dtypes(include="number").describe())

print("\nBEFORE ANALYSIS CHECKS COMPLETED.")
print("Quality log saved in outputs folder.")


age_sex.csv
Columns: ['district_id', 'age_start', 'sex', 'population_2025', 'baseline_deaths_2025', 'arrivals_2025', 'departures_2025', 'births_to_women_2025', 'survival_5yr']
Data types:
 district_id              object
age_start                 int64
sex                      object
population_2025           int64
baseline_deaths_2025      int64
arrivals_2025             int64
departures_2025           int64
births_to_women_2025      int64
survival_5yr            float64
dtype: object

projection_scenarios.csv
Columns: ['scenario', 'fertility_multiplier', 'arrivals_multiplier', 'departures_multiplier', 'newborn_survival_f', 'newborn_survival_m']
Data types:
 scenario                  object
fertility_multiplier     float64
arrivals_multiplier      float64
departures_multiplier    float64
newborn_survival_f       float64
newborn_survival_m       float64
dtype: object

districts.csv
Columns: ['district_id', 'district_name', 'region', 'population_2020', 'population_2026', 'births_2025',


The preliminary data-quality checks were executed successfully, generating a comprehensive log (before_analysis_quality_log.csv). The output confirms the structural integrity of the synthetic datasets prior to analysis:

· Row Counts and Keys: The unique identifier checks confirmed that district_id and other primary keys contain no duplicates, ensuring safe join cardinality and preventing row duplication during data merges.
· Valid Ranges and Missing Indicators: The numerical range checks (describe()) validated the data. For example, five-year survival probabilities fall strictly between 0 and 1, interval widths (ax) align with standard demographic assumptions (ranging from ~0.31 for infancy to 2.5), and scenario multipliers are appropriately centered around the base value of 1.0. No impossible values (e.g., negative populations or births) were detected. Missing values were systematically identified and logged.
· Scope Difference: Acknowledged is the scope difference between districts.csv and age_sex.csv. District-level deaths represent total registered events within the geographic boundary, whereas baseline deaths only include those successfully matched to specific age-sex cohorts. They are not forced to reconcile perfectly.
· Data Retention: All original CSV files remain untouched in the source folder. Cleaning decisions (such as the median imputation for the two missing piped-water indicators) are documented and executed downstream in the relevant analytical sections to retain all 24 districts.